# 05 · Extracción y ponderación de características

Bolsa de palabras con unigramas y bigramas, stopwords, stemming y ponderación TO y TF-IDF sobre `texto_limpio`. Las funciones están en `caracteristicas.py`, que también usa el notebook de clasificación.

## 0. Configuración

In [1]:
from pathlib import Path

import pandas as pd

import caracteristicas as car

BASE = Path.cwd()
DATOS = BASE / "datos"
RESULTADOS = BASE / "resultados"
RESULTADOS.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 200)

corpus = pd.read_csv(DATOS / "corpus_limpio.csv", keep_default_na=False)
textos = corpus["texto_limpio"]
print(corpus.shape)

(5150, 6)


## 1. Tokenización y bolsa de palabras

El texto se separa por espacios y los bigramas se forman con cada token y el siguiente. Ejemplo con un documento:

In [2]:
ejemplo = corpus.iloc[0]
tokens = ejemplo["texto_limpio"].split()
terminos = car.analizar(ejemplo["texto_limpio"])
bigramas = terminos[len(tokens):]

print(ejemplo["doc_id"], "|", ejemplo["clase"])
print("Texto:", ejemplo["texto_limpio"])
print(f"\n{len(tokens)} unigramas:", tokens[:12], "...")
print(f"{len(bigramas)} bigramas:", bigramas[:8], "...")

curado_2501 | Verdad
Texto: el de los alemanes esta en contra de la extradicion de puigdemont a españa una encuesta de diario die welt revela que son los votantes de la izquierda y de la ultraderecha los que mas rechazan que el expresident de la generalitat sea entregado a españa donde se le acusa de rebelion y malversacion de fondos publicos

57 unigramas: ['el', 'de', 'los', 'alemanes', 'esta', 'en', 'contra', 'de', 'la', 'extradicion', 'de', 'puigdemont'] ...
56 bigramas: ['el de', 'de los', 'los alemanes', 'alemanes esta', 'esta en', 'en contra', 'contra de', 'de la'] ...


## 2. Stopwords

Lista de NLTK en español, sin tildes como el texto limpio, y eliminación de los términos que aparecen en menos de 3 documentos (`min_df=3`).

In [3]:
print(f"Stopwords de NLTK sin tildes: {len(car.STOPWORDS)}")
print(sorted(car.STOPWORDS)[:40], "...")

Stopwords de NLTK sin tildes: 306
['a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre', 'era', 'erais', 'eramos', 'eran', 'eras', 'eres', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estabais', 'estabamos'] ...


## 3. Stemming

Snowball Stemmer de NLTK para español.

In [4]:
palabras = ["gobierno", "gobernar", "elecciones", "eleccion", "presidente", "presidenta",
            "politica", "politicos", "corrupcion", "corrupto"]
pd.DataFrame({"palabra": palabras, "raíz": [car.raiz(p) for p in palabras]})

,palabra,raíz
0,gobierno,gobiern
1,gobernar,gobern
2,elecciones,eleccion
3,eleccion,eleccion
4,presidente,president
5,presidenta,president
6,politica,polit
7,politicos,polit
8,corrupcion,corrupcion
9,corrupto,corrupt


## 4. Reducciones sobre un mismo documento

In [5]:
reducciones = {
    "Sin ninguna técnica": dict(quitar_stopwords=False, aplicar_stemming=False),
    "Solo stopwords": dict(quitar_stopwords=True, aplicar_stemming=False),
    "Solo stemming": dict(quitar_stopwords=False, aplicar_stemming=True),
    "Stopwords + stemming": dict(quitar_stopwords=True, aplicar_stemming=True),
}

filas = []
for nombre, opciones in reducciones.items():
    terminos = car.analizar(ejemplo["texto_limpio"], **opciones)
    n_unigramas = len([t for t in terminos if " " not in t])
    filas.append({"reducción": nombre, "unigramas": n_unigramas, "bigramas": len(terminos) - n_unigramas,
                  "primeros términos": ", ".join(terminos[:10])})
pd.DataFrame(filas)

,reducción,unigramas,bigramas,primeros términos
0,Sin ninguna técnica,57,56,"el, de, los, alemanes, esta, en, contra, de, la, extradicion"
1,Solo stopwords,22,21,"alemanes, extradicion, puigdemont, españa, encuesta, diario, die, welt, revela, votantes"
2,Solo stemming,57,56,"el, de, los, aleman, esta, en, contr, de, la, extradicion"
3,Stopwords + stemming,22,21,"aleman, extradicion, puigdemont, españ, encuest, diari, die, welt, revel, votant"


## 5. Tamaño del vocabulario

Términos distintos en todo el corpus según la reducción aplicada.

In [6]:
def tamano_vocabulario(quitar_stopwords, aplicar_stemming, min_df):
    vectorizador = car.CountVectorizer(
        analyzer=lambda texto: car.analizar(texto, quitar_stopwords, aplicar_stemming), min_df=min_df)
    vocabulario = vectorizador.fit(textos).get_feature_names_out()
    bigramas = sum(" " in termino for termino in vocabulario)
    return {"unigramas": len(vocabulario) - bigramas, "bigramas": bigramas, "total": len(vocabulario)}


pasos = [
    ("Bolsa de palabras (sin ninguna técnica)", False, False, 1),
    ("+ lista de stopwords de NLTK", True, False, 1),
    ("+ frecuencia mínima (min_df = 3)", True, False, car.FRECUENCIA_MINIMA),
    ("+ stemming", True, True, car.FRECUENCIA_MINIMA),
]
efecto = pd.DataFrame([{"paso": nombre, **tamano_vocabulario(sw, st, m)} for nombre, sw, st, m in pasos])
efecto["reducción vs. sin técnica (%)"] = (100 * (1 - efecto["total"] / efecto["total"].iloc[0])).round(1)
efecto.to_csv(RESULTADOS / "05_efecto_de_cada_paso.csv", index=False, encoding="utf-8")
efecto

,paso,unigramas,bigramas,total,reducción vs. sin técnica (%)
0,Bolsa de palabras (sin ninguna técnica),18690,105507,124197,0.0
1,+ lista de stopwords de NLTK,18495,108154,126649,-2.0
2,+ frecuencia mínima (min_df = 3),7205,4001,11206,91.0
3,+ stemming,4678,5364,10042,91.9


In [7]:
configs_reduccion = [
    ("Sin ninguna técnica", False, False),
    ("Solo stopwords", True, False),
    ("Solo stemming", False, True),
    ("Stopwords + stemming", True, True),
]
vocabularios = pd.DataFrame([
    {"reducción": nombre, "STOPWORDS": sw, "STEMMING": st,
     **tamano_vocabulario(sw, st, car.FRECUENCIA_MINIMA if sw else 1)}
    for nombre, sw, st in configs_reduccion
])
vocabularios["reducción vs. sin técnica (%)"] = (100 * (1 - vocabularios["total"] / vocabularios["total"].iloc[0])).round(1)
vocabularios.to_csv(RESULTADOS / "05_vocabulario_por_reduccion.csv", index=False, encoding="utf-8")
vocabularios

,reducción,STOPWORDS,STEMMING,unigramas,bigramas,total,reducción vs. sin técnica (%)
0,Sin ninguna técnica,False,False,18690,105507,124197,0.0
1,Solo stopwords,True,False,7205,4001,11206,91.0
2,Solo stemming,False,True,9570,91333,100903,18.8
3,Stopwords + stemming,True,True,4678,5364,10042,91.9


La lista de stopwords sola aumenta un poco los términos porque, al quitar palabras, aparecen bigramas nuevos. La mayor reducción viene del filtro de frecuencia.

## 6. Ponderación: TO y TF-IDF

Pesos de los términos del documento de ejemplo con cada ponderación.

In [8]:
vect_to = car.construir_vectorizador("TO", quitar_stopwords=True, aplicar_stemming=True)
vect_tfidf = car.construir_vectorizador("TF-IDF", quitar_stopwords=True, aplicar_stemming=True)
X_to = vect_to.fit_transform(textos)
X_tfidf = vect_tfidf.fit_transform(textos)

terminos = vect_to.get_feature_names_out()
fila = 0   # el documento de ejemplo es el primero del corpus
pesos = pd.DataFrame({
    "término": terminos[X_to[fila].indices],
    "TO": X_to[fila].data,
    "TF-IDF": X_tfidf[fila, X_to[fila].indices].toarray().ravel().round(3),
    "documentos donde aparece": (X_to[:, X_to[fila].indices] > 0).sum(axis=0).A1,
}).sort_values("TF-IDF", ascending=False)
pesos.head(10)

,término,TO,TF-IDF,documentos donde aparece
21,rebelion malvers,1,0.272,4
20,acus rebelion,1,0.260,6
3,españ,2,0.257,327
1,extradicion,1,0.256,7
0,aleman,1,0.252,8
22,malvers fond,1,0.252,8
23,fond public,1,0.237,13
19,expresident generalitat,1,0.226,18
15,rebelion,1,0.224,19
16,malvers,1,0.217,24


In [9]:
pesos.tail(5)

,término,TO,TF-IDF,documentos donde aparece
11,expresident,1,0.152,166
8,izquierd,1,0.143,212
10,rechaz,1,0.138,245
14,acus,1,0.133,288
18,public,1,0.114,506


## 7. Las 8 configuraciones

Tamaño de la matriz de características de cada configuración de la sección 10.

In [10]:
filas = []
for config in car.CONFIGURACIONES:
    vectorizador = car.construir_vectorizador(config["POND"], config["STOPWORDS"], config["STEMMING"])
    X = vectorizador.fit_transform(textos)
    filas.append({**config,
                  "documentos": X.shape[0],
                  "características": X.shape[1],
                  "% celdas distintas de 0": round(100 * X.nnz / (X.shape[0] * X.shape[1]), 3),
                  "peso máximo": round(X.max(), 3)})
matrices = pd.DataFrame(filas)
matrices.to_csv(RESULTADOS / "05_configuraciones.csv", index=False, encoding="utf-8")
matrices

,#,POND,STOPWORDS,STEMMING,documentos,características,% celdas distintas de 0,peso máximo
0,1,TO,False,False,5150,124197,0.074,17.000
1,2,TO,True,False,5150,11206,0.260,7.000
2,3,TO,False,True,5150,100903,0.091,17.000
3,4,TO,True,True,5150,10042,0.315,7.000
4,5,TF-IDF,False,False,5150,124197,0.074,0.492
5,6,TF-IDF,True,False,5150,11206,0.260,0.702
6,7,TF-IDF,False,True,5150,100903,0.091,0.551
7,8,TF-IDF,True,True,5150,10042,0.315,0.728
